# MegaBrain no Google Colab — tudo em um lugar

Este notebook roda o projeto inteiro numa GPU do Colab, usando o código de `src/` do repositório:

| Parte | O que faz | Tempo numa GPU T4 |
|---|---|---|
| 1 | Prepara o ambiente e baixa o código | 1 min |
| 2 | Carrega o GPT-2 124M no nosso `GPTModel` e gera texto | 1 min |
| 3 | **Seus PDFs → corpus de treino** (extração e limpeza do texto) | depende dos PDFs |
| 4 | **Continua o pré-treino** do GPT-2 no texto dos PDFs | depende do corpus |
| 5 | Fine-tuning de **classificação** (spam × não spam) | poucos minutos |
| 6 | Fine-tuning de **instruções** (pergunta → resposta) | poucos minutos |
| 7 | **Teste os três modelos** com os seus textos | segundos |
| 8 | Salva os modelos no Google Drive | 1 min |

**Antes de começar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.
Depois, *Ambiente de execução → Executar tudo*. As partes 4, 5 e 6 são independentes entre si: dá para
rodar só a que interessa, desde que as partes 1 e 2 tenham rodado.

Os tempos acima são estimativas: o notebook foi verificado de ponta a ponta em CPU, no modo rápido, e não
numa GPU do Colab.

## 1. Ambiente

Ajuste aqui o que for preciso. O resto do notebook lê estas variáveis.

In [ ]:
# --- configuração ----------------------------------------------------------
REPOSITORIO = "https://github.com/Davyjosias01/MegaBrain.git"
BRANCH = "sprint-6"            # depois que a sprint for mesclada, troque para "develop"

USAR_DRIVE = True              # guarda PDFs, corpus e modelos no seu Google Drive
PASTA_NO_DRIVE = "MegaBrain"   # pasta dentro de "Meu Drive"; coloque os PDFs em MegaBrain/pdfs

MODO_RAPIDO = False            # True = poucos dados e poucos passos, só para ver se tudo roda

# Parte 4 — continuação do pré-treino no texto dos PDFs
EPOCAS_PDFS = 3
CONTEXTO_PDFS = 256            # tokens por janela de treino
LOTE_PDFS = 8

# Parte 5 — classificação
EPOCAS_CLASSIFICACAO = 5
BLOCOS_CLASSIFICACAO = 1       # últimos blocos treináveis; None = modelo inteiro

# Parte 6 — instruções
TAMANHO_INSTRUCOES = "small"   # "small" = GPT-2 124M; "medium" = 355M (o do livro; não testado aqui)
EPOCAS_INSTRUCOES = 2
BLOCOS_INSTRUCOES = None       # None = modelo inteiro, como no livro

In [ ]:
import gc
import os
import subprocess
import sys
import time
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
MODO_RAPIDO = MODO_RAPIDO or os.environ.get("MEGABRAIN_RAPIDO") == "1"

if NO_COLAB:
    RAIZ = Path("/content/MegaBrain")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPOSITORIO, str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken", "safetensors", "pypdf"], check=True)
else:                          # fora do Colab: usa o repositório onde o notebook está
    RAIZ = Path.cwd().parent if (Path.cwd().parent / "src").exists() else Path.cwd()

sys.path.insert(0, str(RAIZ / "src"))

if NO_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    TRABALHO = Path("/content/drive/MyDrive") / PASTA_NO_DRIVE
else:
    TRABALHO = RAIZ / "modelos" / "colab"
PASTA_PDFS = Path(os.environ.get("MEGABRAIN_PDFS", TRABALHO / "pdfs"))
PASTA_PDFS.mkdir(parents=True, exist_ok=True)

import torch

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("código     :", RAIZ)
print("trabalho   :", TRABALHO)
print("PDFs em    :", PASTA_PDFS)
print("dispositivo:", DISPOSITIVO, f"({torch.cuda.get_device_name(0)})" if DISPOSITIVO == "cuda" else
      "— sem GPU: ative-a no menu Ambiente de execução, ou use MODO_RAPIDO = True")
print("modo rápido:", MODO_RAPIDO)

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader

from checkpoint import carregar_checkpoint, salvar_checkpoint
from classificacao import (
    PASTA_SPAM, DatasetDeClassificacao, adaptar_para_classificacao, carregar_classificador, classificar_texto,
    contar_treinaveis, criar_loaders_classificacao, metricas_de_classificacao, preparar_base_spam,
    treinar_classificador,
)
from dados import carregar_texto, criar_loaders_treino_validacao
from geracao import gerar, ids_para_texto, texto_para_ids
from instrucoes import (
    avaliar_respostas, criar_loaders_instrucoes, definir_treinaveis, montar_prompt, preparar_instrucoes, responder,
)
from pdf_para_dataset import listar_pdfs, montar_corpus, resumo_do_corpus
from pesos_pretreinados import carregar_gpt2_pretreinado
from tokenizacao import TokenizadorBPE
from treinamento import perda_do_loader, perplexidade, treinar_modelo

tok = TokenizadorBPE()
SEMENTE = 123


def liberar():
    # Devolve a memória de modelos já apagados com `del` (cada parte usa um modelo por vez).
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def escrever(modelo, contexto, tokens=60, temperatura=0.8, top_k=40, semente=SEMENTE):
    """Continua `contexto` com o modelo e devolve o texto."""
    torch.manual_seed(semente)
    ids = texto_para_ids(contexto, tok).to(DISPOSITIVO)
    saida = gerar(modelo.eval(), ids, tokens, modelo.cfg["context_length"], temperatura=temperatura, top_k=top_k)
    return ids_para_texto(saida, tok)

print("módulos de src/ carregados")

## 2. O GPT-2 pré-treinado no nosso `GPTModel`

Os pesos publicados pela OpenAI (548 MB) são baixados uma vez e copiados para o modelo que escrevemos nas
Sprints 2 a 4. É o ponto de partida de todos os treinos abaixo.

In [ ]:
gpt2 = carregar_gpt2_pretreinado("small", dispositivo=DISPOSITIVO)
print(f"{sum(p.numel() for p in gpt2.parameters()):,} parâmetros\n")
for contexto in ["Every effort moves you", "The best way to grow tomatoes is"]:
    print(escrever(gpt2, contexto, tokens=20 if MODO_RAPIDO else 50), "\n" + "-" * 80)
del gpt2
liberar()

## 3. Seus PDFs → corpus de treino

Coloque os PDFs na pasta mostrada na Parte 1 (`MegaBrain/pdfs` no seu Drive; subpastas são aceitas) e rode a
célula. Se a pasta estiver vazia, o Colab abre um botão para enviar arquivos do computador.

O que acontece com cada PDF (`src/pdf_para_dataset.py`):

1. o texto de cada página é extraído;
2. cabeçalhos, rodapés e números de página são removidos, a hifenização de fim de linha é desfeita e as
   linhas viram parágrafos;
3. os documentos são reunidos num único `corpus.txt`, separados por `<|endoftext|>`.

Três coisas para saber:

- **PDF escaneado não tem texto.** Um arquivo que é só imagem é ignorado e aparece na lista de ignorados;
  lê-lo exigiria OCR, que este projeto não faz.
- **Isto é um dataset de texto corrido**, que serve para o modelo aprender o vocabulário e o estilo dos
  seus textos (Parte 4). Não é um dataset de perguntas e respostas: para isso alguém precisa escrever os
  pares, no formato da Parte 6.
- **Você responde pelo que coloca aqui.** O modelo passa a reproduzir o que leu: use textos que você pode
  usar (direitos autorais) e cujo conteúdo você quer ver reproduzido. Mantenha os PDFs e o corpus no seu
  Drive, fora do repositório público.

Se não houver nenhum PDF, o notebook usa o conto `the-verdict.txt` do repositório como exemplo, para que as
partes seguintes rodem.

In [ ]:
if NO_COLAB and not listar_pdfs(PASTA_PDFS):
    from google.colab import files
    print("Nenhum PDF em", PASTA_PDFS, "— envie arquivos agora (ou cancele para usar o exemplo):")
    for nome, conteudo in files.upload().items():
        (PASTA_PDFS / nome).write_bytes(conteudo)

ARQUIVO_CORPUS = TRABALHO / "corpus.txt"
pdfs = listar_pdfs(PASTA_PDFS)
if pdfs:
    print(f"{len(pdfs)} PDFs encontrados\n")
    relatorio = montar_corpus(pdfs, ARQUIVO_CORPUS, max_paginas=5 if MODO_RAPIDO else None)
    corpus, ORIGEM_DO_CORPUS = relatorio["texto"], f"{len(relatorio['documentos'])} PDFs"
else:
    corpus, ORIGEM_DO_CORPUS = carregar_texto(), "exemplo (the-verdict.txt): nenhum PDF foi fornecido"
if MODO_RAPIDO:
    corpus = corpus[:6000]

print("\ncorpus:", ORIGEM_DO_CORPUS)
resumo = resumo_do_corpus(corpus, tok, CONTEXTO_PDFS)
for chave, valor in resumo.items():
    print(f"  {chave:<20}: {valor:,.2f}" if isinstance(valor, float) else f"  {chave:<20}: {valor:,}")
print("\ninício do corpus:\n" + corpus[:500])

**Quanto texto é suficiente?** Olhe `janelas_de_treino` acima. Com poucas centenas de janelas o modelo
decora o texto (foi o que aconteceu na Sprint 5, com 20 janelas); com dezenas de milhares ele aprende o
estilo. Um livro de 300 páginas dá cerca de 150 mil tokens, ou 600 janelas de 256 tokens.

## 4. Continuar o pré-treino no texto dos PDFs

Mesmo laço da Sprint 5 (`treinar_modelo`), mas partindo do GPT-2 em vez de pesos aleatórios. O corpus é
dividido em 90% para treino e 10% para validação; a perda de validação diz se o modelo está aprendendo o
assunto ou só decorando.

In [ ]:
contexto_pdfs = 64 if MODO_RAPIDO else CONTEXTO_PDFS
loader_treino, loader_validacao = criar_loaders_treino_validacao(
    corpus, batch_size=2 if MODO_RAPIDO else LOTE_PDFS, max_length=contexto_pdfs)
print(f"{len(loader_treino)} lotes de treino, {len(loader_validacao)} de validação")
if len(loader_treino) == 0 or len(loader_validacao) == 0:
    raise ValueError("corpus pequeno demais para este contexto: junte mais texto ou reduza CONTEXTO_PDFS.")

# O primeiro trecho do corpus de validação serve de contexto para comparar antes e depois.
CONTEXTO_DE_TESTE = " ".join(corpus[int(len(corpus) * 0.9):].split()[1:13])  # pula a palavra cortada

modelo_pdfs = carregar_gpt2_pretreinado("small", dispositivo=DISPOSITIVO)
if MODO_RAPIDO:
    definir_treinaveis(modelo_pdfs, 1)
perda_antes = perda_do_loader(loader_validacao, modelo_pdfs, DISPOSITIVO)
texto_antes = escrever(modelo_pdfs, CONTEXTO_DE_TESTE, tokens=20 if MODO_RAPIDO else 60)
print(f"antes : perda de validação {perda_antes:.3f} (perplexidade {perplexidade(perda_antes):.1f})")

torch.manual_seed(SEMENTE)
otimizador = torch.optim.AdamW([p for p in modelo_pdfs.parameters() if p.requires_grad], lr=5e-5, weight_decay=0.1)
inicio = time.perf_counter()
historico_pdfs = treinar_modelo(
    modelo_pdfs, loader_treino, loader_validacao, otimizador, DISPOSITIVO,
    num_epocas=1 if MODO_RAPIDO else EPOCAS_PDFS,
    frequencia_avaliacao=max(1, len(loader_treino) // 4), lotes_avaliacao=5)
perda_depois = perda_do_loader(loader_validacao, modelo_pdfs, DISPOSITIVO)
print(f"\ndepois: perda de validação {perda_depois:.3f} (perplexidade {perplexidade(perda_depois):.1f}) "
      f"em {(time.perf_counter() - inicio) / 60:.1f} min")

In [ ]:
print("CONTEXTO:", CONTEXTO_DE_TESTE, "\n")
print("GPT-2 original:\n", texto_antes, "\n")
print("depois de ler os seus textos:\n", escrever(modelo_pdfs, CONTEXTO_DE_TESTE, tokens=20 if MODO_RAPIDO else 60))

salvar_checkpoint(TRABALHO / "gpt2-pdfs.pt", modelo_pdfs, cfg_treino={"corpus": ORIGEM_DO_CORPUS}, historico=historico_pdfs)
print("\nmodelo salvo em", TRABALHO / "gpt2-pdfs.pt")
del modelo_pdfs
liberar()

**Como ler o resultado.** Se a perda de **validação** caiu, o modelo aprendeu algo que vale para texto que
ele não viu. Se a de treino cai e a de validação sobe, ele está decorando: use menos épocas ou mais texto.

## 5. Fine-tuning de classificação (spam × não spam)

Troca a cabeça de saída por uma de 2 classes e ajusta o fim do modelo na *SMS Spam Collection* (914
mensagens de treino). Em CPU este treino levou 40 minutos e chegou a 98,1% no teste.

In [ ]:
preparar_base_spam(semente=SEMENTE)
if MODO_RAPIDO:
    partes = [pd.read_csv(PASTA_SPAM / f"{n}.csv").head(k) for n, k in (("treino", 32), ("validacao", 16), ("teste", 16))]
    conjuntos = [DatasetDeClassificacao(p, tok, max_length=48) for p in partes]
    lt = DataLoader(conjuntos[0], batch_size=8, shuffle=True, drop_last=True,
                    generator=torch.Generator().manual_seed(SEMENTE))
    lv, lte = DataLoader(conjuntos[1], batch_size=8), DataLoader(conjuntos[2], batch_size=8)
else:
    lt, lv, lte = criar_loaders_classificacao(tokenizador=tok, batch_size=8, semente=SEMENTE)
COMPRIMENTO_MENSAGEM = lt.dataset.max_length

classificador = carregar_gpt2_pretreinado("small", dispositivo=DISPOSITIVO)
torch.manual_seed(SEMENTE)
adaptar_para_classificacao(classificador, 2, blocos_treinaveis=BLOCOS_CLASSIFICACAO)
print("treináveis: {:,} de {:,}".format(*contar_treinaveis(classificador)))
antes = metricas_de_classificacao(lte, classificador, DISPOSITIVO)

otimizador = torch.optim.AdamW([p for p in classificador.parameters() if p.requires_grad], lr=5e-5, weight_decay=0.1)
inicio = time.perf_counter()
historico_classificador = treinar_classificador(
    classificador, lt, lv, otimizador, DISPOSITIVO, num_epocas=1 if MODO_RAPIDO else EPOCAS_CLASSIFICACAO)
depois = metricas_de_classificacao(lte, classificador, DISPOSITIVO)
print(f"\ntreino em {(time.perf_counter() - inicio) / 60:.1f} min — conjunto de teste:")
display(pd.DataFrame([antes, depois], index=["antes do ajuste", "depois do ajuste"]).round(4))

salvar_checkpoint(TRABALHO / "classificador-spam.pt", classificador,
                  cfg_treino={"max_length": COMPRIMENTO_MENSAGEM}, historico=historico_classificador)
del classificador
liberar()

## 6. Fine-tuning de instruções

O modelo continua prevendo o próximo token; o que muda são os dados: 913 pares instrução → resposta no
formato Alpaca. Numa GPU dá para ajustar o **modelo inteiro** (`BLOCOS_INSTRUCOES = None`), o que em CPU
não coube — lá foram 6 blocos, com 7,5% de respostas idênticas à esperada.

Para treinar com as **suas** perguntas e respostas, monte uma lista no mesmo formato e junte ao treino:

```python
meus_exemplos = [
    {"instruction": "What is the capital of Brazil?", "input": "", "output": "The capital of Brazil is Brasília."},
]
treino = treino + meus_exemplos
```

In [ ]:
treino, validacao, teste = preparar_instrucoes()
if MODO_RAPIDO:
    treino, validacao, teste = treino[:16], validacao[:8], teste[:4]
print(f"treino {len(treino)} | validação {len(validacao)} | teste {len(teste)}")

tamanho = "small" if MODO_RAPIDO else TAMANHO_INSTRUCOES
assistente = carregar_gpt2_pretreinado(tamanho, dispositivo=DISPOSITIVO)
treinaveis, total = definir_treinaveis(assistente, 1 if MODO_RAPIDO else BLOCOS_INSTRUCOES)
print(f"GPT-2 {tamanho}: {treinaveis:,} parâmetros treináveis de {total:,}")

torch.manual_seed(SEMENTE)
lt, lv, _ = criar_loaders_instrucoes(treino, validacao, teste, tok, batch_size=8,
                                     max_length=assistente.cfg["context_length"], semente=SEMENTE)
respostas_antes = [responder(e, assistente, tok, dispositivo=DISPOSITIVO, max_novos_tokens=48) for e in teste[:20]]

otimizador = torch.optim.AdamW([p for p in assistente.parameters() if p.requires_grad], lr=5e-5, weight_decay=0.1)
inicio = time.perf_counter()
historico_instrucoes = treinar_modelo(
    assistente, lt, lv, otimizador, DISPOSITIVO, num_epocas=1 if MODO_RAPIDO else EPOCAS_INSTRUCOES,
    frequencia_avaliacao=max(1, len(lt) // 5), lotes_avaliacao=5)
print(f"\ntreino em {(time.perf_counter() - inicio) / 60:.1f} min")

In [ ]:
assistente.eval()
respostas_depois = [responder(e, assistente, tok, dispositivo=DISPOSITIVO, max_novos_tokens=48) for e in teste]
esperadas = [e["output"] for e in teste]
display(pd.DataFrame({
    "antes do ajuste": avaliar_respostas(list(zip(respostas_antes, esperadas[:len(respostas_antes)]))),
    "depois do ajuste": avaliar_respostas(list(zip(respostas_depois, esperadas))),
}).round(3).T)
print("(antes: os", len(respostas_antes), "primeiros exemplos de teste; depois: todos os", len(teste), ")\n")

for exemplo, resposta in list(zip(teste, respostas_depois))[:8]:
    print("instrução:", exemplo["instruction"], exemplo["input"])
    print("  esperada:", exemplo["output"])
    print("  modelo  :", resposta, "\n")

salvar_checkpoint(TRABALHO / "instrucoes-gpt2.pt", assistente, cfg_treino={"tamanho": tamanho}, historico=historico_instrucoes)
del assistente
liberar()

**Leia as respostas, não só a tabela.** O F1 de palavras mede semelhança de redação, não correção: uma
resposta no molde certo com o fato errado tem F1 alto. A correspondência exata é o piso da taxa de acerto.

## 7. Teste os modelos

Edite as listas e rode a célula. Os três modelos são carregados dos arquivos salvos nas partes anteriores;
os que não existirem são pulados. Os modelos só entendem **inglês**.

In [ ]:
TEXTOS_PARA_CONTINUAR = [CONTEXTO_DE_TESTE if "CONTEXTO_DE_TESTE" in globals() else "Every effort moves you"]
MENSAGENS = [
    "Congratulations! You won a free iPhone. Click here to claim your prize now.",
    "Hey, are we still meeting for dinner tonight?",
]
PERGUNTAS = [
    ("What is the opposite of cold?", ""),
    ("Rewrite the sentence in the past tense.", "She walks to school every day."),
]

arquivo = TRABALHO / "gpt2-pdfs.pt"
if arquivo.exists():
    modelo, _ = carregar_checkpoint(arquivo, DISPOSITIVO)
    print("=== modelo que leu os seus textos ===")
    for contexto in TEXTOS_PARA_CONTINUAR:
        print(escrever(modelo, contexto, tokens=20 if MODO_RAPIDO else 80), "\n")
    del modelo
    liberar()

arquivo = TRABALHO / "classificador-spam.pt"
if arquivo.exists():
    modelo, dados = carregar_classificador(arquivo, DISPOSITIVO)
    print("=== classificador de spam ===")
    for mensagem in MENSAGENS:
        classe, probs = classificar_texto(mensagem, modelo, tok, DISPOSITIVO, max_length=dados["cfg_treino"]["max_length"])
        print(f"{classe:>9} ({probs[classe]:.0%}) <- {mensagem}")
    del modelo
    liberar()

arquivo = TRABALHO / "instrucoes-gpt2.pt"
if arquivo.exists():
    modelo, _ = carregar_checkpoint(arquivo, DISPOSITIVO)
    print("\n=== modelo de instruções ===")
    for pergunta, entrada in PERGUNTAS:
        print("pergunta:", pergunta, entrada)
        print("resposta:", responder(pergunta, modelo, tok, entrada=entrada, dispositivo=DISPOSITIVO, max_novos_tokens=48), "\n")
    del modelo
    liberar()

Para conversar em sequência com o modelo de instruções, rode a célula abaixo e digite as perguntas
(linha vazia encerra). Ela fica desligada por padrão para não travar o *Executar tudo*.

In [ ]:
CONVERSAR = False
if CONVERSAR and (TRABALHO / "instrucoes-gpt2.pt").exists():
    modelo, _ = carregar_checkpoint(TRABALHO / "instrucoes-gpt2.pt", DISPOSITIVO)
    while True:
        pergunta = input("pergunta (em inglês; vazio para sair): ").strip()
        if not pergunta:
            break
        print("->", responder(pergunta, modelo, tok, dispositivo=DISPOSITIVO, max_novos_tokens=64), "\n")
    del modelo
    liberar()

## 8. Onde ficaram os arquivos

Com `USAR_DRIVE = True`, tudo já está no seu Google Drive, na pasta `MegaBrain`, e continua lá quando a
sessão do Colab acabar. Cada modelo tem de 500 a 660 MB.

Para usar um desses modelos no seu computador, baixe o `.pt` do Drive para a pasta `modelos/` do repositório
e carregue com `carregar_checkpoint` (ou `carregar_classificador`, para o de spam).

In [ ]:
for arquivo in sorted(TRABALHO.glob("*")):
    if arquivo.is_file():
        print(f"{arquivo.stat().st_size / 1e6:>8.1f} MB  {arquivo}")